# SpannerGraphStorage Interactive Test Notebook

This notebook provides an interactive, step-by-step test suite for `SpannerGraphStorage`.
Each cell can be run independently to test specific operations.

## Prerequisites

1. **Google Cloud authentication**
   ```bash
   gcloud auth application-default login
   ```

2. **Enable required APIs**
   ```bash
   gcloud services enable spanner.googleapis.com aiplatform.googleapis.com cloudresourcemanager.googleapis.com
   ```

3. **Create a Spanner instance and database** (if not already created)
   ```bash
   export SPANNER_INSTANCE="pathrag"
   export SPANNER_DATABASE="pathrag_db"
   export SPANNER_REGION="us-central1"

   gcloud spanner instances create $SPANNER_INSTANCE \
     --config=regional-$SPANNER_REGION \
     --description="Spanner instance for PathRAG" \
     --nodes=1 \
     --edition=ENTERPRISE

   gcloud spanner databases create $SPANNER_DATABASE \
     --instance=$SPANNER_INSTANCE
   ```

4. **Install dependencies**
   ```bash
   pip install -e .
   pip install google-cloud-spanner
   ```

5. **Configure `.env` file** in `examples/spanner/` (see `.env.example`)

> **Note:** `SpannerGraphStorage` automatically creates the required tables and property graph DDL on first initialization. No manual schema setup is needed.

---
## Setup: Load environment and create storage

In [ ]:
import json
import os

from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

GOOGLE_CLOUD_PROJECT = os.environ.get("GOOGLE_CLOUD_PROJECT")
SPANNER_INSTANCE = os.environ.get("SPANNER_INSTANCE")
SPANNER_DATABASE = os.environ.get("SPANNER_DATABASE")

print(f"Project:  {GOOGLE_CLOUD_PROJECT}")
print(f"Instance: {SPANNER_INSTANCE}")
print(f"Database: {SPANNER_DATABASE}")

assert SPANNER_INSTANCE and SPANNER_DATABASE, (
    "SPANNER_INSTANCE and SPANNER_DATABASE must be set in .env or environment variables.\n"
    "See examples/spanner/.env.example"
)

In [ ]:
from PathRAG.storage.spanner import SpannerGraphStorage

def create_storage(namespace: str = "test"):
    """Create a SpannerGraphStorage instance for testing."""
    return SpannerGraphStorage(
        namespace=namespace,
        global_config={
            "spanner_project_id": GOOGLE_CLOUD_PROJECT,
            "spanner_instance_id": SPANNER_INSTANCE,
            "spanner_database_id": SPANNER_DATABASE,
        },
    )

NAMESPACE = "test_crud"
graph = create_storage(NAMESPACE)
print(f"SpannerGraphStorage created (namespace={NAMESPACE})")

---
## Step 1: Node CRUD Operations

Insert, read, update, and list nodes in the graph.

### 1-1. Insert nodes

In [ ]:
await graph.upsert_node("APPLE", {
    "entity_type": "ORGANIZATION",
    "description": "American multinational technology company",
    "source_id": "chunk-001",
})
await graph.upsert_node("STEVE JOBS", {
    "entity_type": "PERSON",
    "description": "Co-founder of Apple Inc.",
    "source_id": "chunk-001",
})
await graph.upsert_node("TIM COOK", {
    "entity_type": "PERSON",
    "description": "Current CEO of Apple Inc.",
    "source_id": "chunk-002",
})
print("Inserted 3 nodes: APPLE, STEVE JOBS, TIM COOK")

### 1-2. Read nodes

In [ ]:
for name in ["APPLE", "STEVE JOBS", "TIM COOK"]:
    exists = await graph.has_node(name)
    data = await graph.get_node(name)
    print(f"  {name}: exists={exists}, data={json.dumps(data)}")

# Check non-existent node
exists = await graph.has_node("NON_EXISTENT")
data = await graph.get_node("NON_EXISTENT")
print(f"  NON_EXISTENT: exists={exists}, data={data}")

### 1-3. List all nodes

In [ ]:
all_nodes = await graph.nodes()
print(f"All nodes ({len(all_nodes)}):")
for n in all_nodes:
    print(f"  {n}")

### 1-4. Update a node

In [ ]:
await graph.upsert_node("APPLE", {
    "entity_type": "ORGANIZATION",
    "description": "Multinational tech company headquartered in Cupertino, CA",
    "source_id": "chunk-001",
})
updated = await graph.get_node("APPLE")
print(f"Updated APPLE: {json.dumps(updated, indent=2)}")

---
## Step 2: Edge CRUD & GQL Traversal

Insert edges, read them via GQL, and check edge existence.

### 2-1. Insert edges

In [ ]:
await graph.upsert_edge("STEVE JOBS", "APPLE", {
    "weight": "3.0",
    "description": "Co-founded Apple in 1976",
    "keywords": "founding, co-founder",
    "source_id": "chunk-001",
})
await graph.upsert_edge("TIM COOK", "APPLE", {
    "weight": "2.5",
    "description": "CEO of Apple since 2011",
    "keywords": "CEO, leadership",
    "source_id": "chunk-002",
})
print("Inserted 2 edges: STEVE JOBS->APPLE, TIM COOK->APPLE")

### 2-2. Read edges (via GQL)

In [ ]:
edge1 = await graph.get_edge("STEVE JOBS", "APPLE")
print(f"STEVE JOBS -> APPLE: {json.dumps(edge1, indent=2)}")

edge2 = await graph.get_edge("TIM COOK", "APPLE")
print(f"TIM COOK -> APPLE: {json.dumps(edge2, indent=2)}")

### 2-3. Check edge existence (via GQL)

In [ ]:
print(f"STEVE JOBS -> APPLE:      {await graph.has_edge('STEVE JOBS', 'APPLE')}")
print(f"APPLE -> STEVE JOBS:      {await graph.has_edge('APPLE', 'STEVE JOBS')}")
print(f"TIM COOK -> STEVE JOBS:   {await graph.has_edge('TIM COOK', 'STEVE JOBS')}")

### 2-4. List all edges

In [ ]:
all_edges = await graph.edges()
print(f"All edges ({len(all_edges)}):")
for src, tgt in all_edges:
    print(f"  {src} -> {tgt}")

---
## Step 3: Graph Traversal (GQL)

Test degree calculations, outgoing/incoming edge traversal, and PageRank approximation.

### 3-1. Node degree

In [ ]:
for name in ["APPLE", "STEVE JOBS", "TIM COOK"]:
    degree = await graph.node_degree(name)
    print(f"  {name}: degree={degree}")

### 3-2. Edge degree

In [ ]:
edge_deg = await graph.edge_degree("STEVE JOBS", "APPLE")
print(f"STEVE JOBS <-> APPLE: edge_degree={edge_deg}")

### 3-3. Outgoing edges

In [ ]:
for name in ["STEVE JOBS", "TIM COOK", "APPLE"]:
    out_edges = await graph.get_node_edges(name)
    print(f"  {name} -> {out_edges}")

### 3-4. Incoming edges

In [ ]:
for name in ["APPLE", "STEVE JOBS"]:
    in_edges = await graph.get_node_in_edges(name)
    print(f"  {name} <- {in_edges}")

### 3-5. PageRank (normalised degree)

In [ ]:
for name in ["APPLE", "STEVE JOBS", "TIM COOK"]:
    pr = await graph.get_pagerank(name)
    print(f"  {name}: pagerank={pr:.4f}")

---
## Step 4: Node Deletion (Cascade)

Delete a node and verify that its edges are also removed.

In [ ]:
nodes_before = await graph.nodes()
edges_before = await graph.edges()
print(f"Before deletion:")
print(f"  Nodes ({len(nodes_before)}): {nodes_before}")
print(f"  Edges ({len(edges_before)}): {edges_before}")

In [ ]:
# Delete STEVE JOBS -- should also remove the edge STEVE JOBS->APPLE
await graph.delete_node("STEVE JOBS")
print("Deleted 'STEVE JOBS'")

nodes_after = await graph.nodes()
edges_after = await graph.edges()
print(f"\nAfter deletion:")
print(f"  Nodes ({len(nodes_after)}): {nodes_after}")
print(f"  Edges ({len(edges_after)}): {edges_after}")
print(f"  STEVE JOBS exists: {await graph.has_node('STEVE JOBS')}")

---
## Step 5: PathRAG Integration with SpannerGraphStorage

End-to-end test: index documents and query using PathRAG with Spanner as the graph backend.

> **Requires:** `GEMINI_API_KEY` or `OPENAI_API_KEY` in `.env`

In [ ]:
def get_llm_config():
    """Resolve LLM configuration from environment variables."""
    if os.environ.get("GEMINI_API_KEY"):
        default_llm = "gemini/gemini-2.5-flash"
        default_embed = "gemini/gemini-embedding-001"
        default_dim = 3072
    elif os.environ.get("OPENAI_API_KEY"):
        default_llm = "gpt-4o-mini"
        default_embed = "text-embedding-3-small"
        default_dim = 1536
    else:
        print("[SKIP] No API key found (GEMINI_API_KEY or OPENAI_API_KEY).")
        return None, None, None, None

    llm_model = os.environ.get("LLM_MODEL_NAME", default_llm)
    embedding_model = os.environ.get("EMBEDDING_MODEL_NAME", default_embed)
    embedding_dim = int(os.environ.get("EMBEDDING_DIM", default_dim))
    tiktoken_model = llm_model

    print(f"LLM Model: {llm_model}")
    print(f"Embedding Model: {embedding_model} (dim={embedding_dim})")
    return llm_model, embedding_model, embedding_dim, tiktoken_model

llm_model, embedding_model, embedding_dim, tiktoken_model = get_llm_config()

In [ ]:
import tempfile
from PathRAG import PathRAG, QueryParam

work_dir = tempfile.mkdtemp(prefix="pathrag_spanner_nb_")
print(f"Working directory: {work_dir}")

rag = PathRAG(
    working_dir=work_dir,
    llm_model_name=llm_model,
    embedding_model_name=embedding_model,
    embedding_dim=embedding_dim,
    tiktoken_model_name=tiktoken_model,
    graph_storage="SpannerGraphStorage",
    addon_params={
        "spanner_project_id": GOOGLE_CLOUD_PROJECT,
        "spanner_instance_id": SPANNER_INSTANCE,
        "spanner_database_id": SPANNER_DATABASE,
    },
    chunk_token_size=200,
    chunk_overlap_token_size=50,
    enable_llm_cache=False,
)
print("PathRAG instance created with SpannerGraphStorage backend")

### 5-1. Index documents

In [ ]:
sample_docs = [
    """
    Apple Inc. is an American multinational technology company
    headquartered in Cupertino, California. Apple was founded on
    April 1, 1976, by Steve Jobs, Steve Wozniak, and Ronald Wayne.
    Tim Cook has been the CEO of Apple since August 2011.
    """,
    """
    Google LLC is an American multinational corporation focusing on
    search engine technology and cloud computing. Sundar Pichai has
    been the CEO of Google since October 2015. Google was founded
    on September 4, 1998, by Larry Page and Sergey Brin.
    """,
]

print(f"Indexing {len(sample_docs)} documents...")
await rag.ainsert(sample_docs)
print("Indexing complete!")

### 5-2. Inspect extracted entities and relationships

In [ ]:
rag_graph = rag.chunk_entity_relation_graph
nodes = await rag_graph.nodes()
edges_list = await rag_graph.edges()

print(f"Extracted entities: {len(nodes)}")
print(f"Extracted relationships: {len(edges_list)}")

print("\n--- Entities ---")
for name in list(nodes)[:10]:
    data = await rag_graph.get_node(name)
    if data:
        print(f"  {name}: type={data.get('entity_type', 'N/A')}")

print("\n--- Relationships ---")
for src, tgt in list(edges_list)[:10]:
    data = await rag_graph.get_edge(src, tgt)
    if data:
        print(f"  {src} --[{data.get('keywords', 'N/A')}]--> {tgt}")

### 5-3. Query

In [ ]:
queries = [
    "What is the relationship between Steve Jobs and Apple?",
    "Who founded Google and what is their background?",
    "Compare the leadership of Apple and Google.",
]

for q in queries:
    print(f"\nQ: {q}")
    response = await rag.aquery(
        q,
        param=QueryParam(mode="hybrid", top_k=10),
    )
    display = response[:300] + "..." if len(response) > 300 else response
    print(f"A: {display}")

---
## Step 6: Cleanup Test Data

Drop the test tables and property graph from Spanner.

> **Warning:** This will permanently delete all test data. Only run this when you are done testing.

In [ ]:
from google.cloud import spanner

client = spanner.Client(
    project=GOOGLE_CLOUD_PROJECT,
    disable_builtin_metrics=True,
)
instance = client.instance(SPANNER_INSTANCE)
database = instance.database(SPANNER_DATABASE)

graph_name = f"pathrag_{NAMESPACE}"
node_table = f"{NAMESPACE}_nodes"
edge_table = f"{NAMESPACE}_edges"

print(f"Dropping property graph '{graph_name}'...")
op = database.update_ddl([f"DROP PROPERTY GRAPH IF EXISTS {graph_name}"])
op.result()

print(f"Dropping edge table '{edge_table}'...")
op = database.update_ddl([f"DROP TABLE IF EXISTS {edge_table}"])
op.result()

print(f"Dropping node table '{node_table}'...")
op = database.update_ddl([f"DROP TABLE IF EXISTS {node_table}"])
op.result()

print("\nCleanup completed!")

---
## References

- [Cloud Spanner Documentation](https://cloud.google.com/spanner/docs)
- [Spanner Graph Overview](https://cloud.google.com/spanner/docs/graph/overview)
- [Spanner GQL Reference](https://cloud.google.com/spanner/docs/reference/standard-sql/graph-query-statements)
- [PathRAG Spanner Setup Guide](../../docs/spanner_setup_guide.md)